> **2026-04-19 CONVENTION WARNING.** This notebook uses the older **UTC-midnight snap convention** (`snap_time = close_ts.floor('D') - N days`). Ship convention per `CLAUDE.md` is **ET-midnight**. The library baseline here uses the v0.1 KDE architecture — shipping 0.2.0 replaces it entirely with Ridge per `plans/plan_ridge_integration.md`. Headline comparison result (Ridge wins library and ship at every snap) is the investigation's verdict and is reflected in `findings/ridge_lambda_investigation.md`. **Do not copy snap/phase timestamp logic from this notebook into new code.**

# Phase-1 three-way: library baseline vs ship stack vs Ridge regression

**Intent:** compare three phase-1 review-count predictors at T-5d, T-4d, T-3d, T-2d, T-1d on the same target-snap set. How much does the ship stack buy us vs. library v0.1? How does Ridge regression compare across horizons (previously tested only at T-3d in path_b_lite §9)?

**Three variants, all evaluated over phase-1 window `(midnight_utc_dbc, snap_dbc_effective]` (I2 convention):**

1. **Library baseline (v0.1):** A1 (default_training_slugs, 20 most recent) + C1 unweighted KDE + D2 floor=0.5 + no bandwidth ceiling + E1 unweighted base_rate + F1 scaling. Raw timestamps, no noon-shift. This is what the library produces out of the box today.
2. **Ship stack (path_b_lite best):** A3 (combined_score α=0.5 σ_gap=8) + C2 weighted KDE + D2+D3 (bw 0.5-0.7d) + E2 weighted base_rate + F1 + G2 midnight snap + H2 noon-shift + I2 phase-1 + K1.
3. **Ridge regression (α=10):** 10 features from the observation window (observed_count, first_review_dbc, target_gap, observed_rate, rate_last_day, rate_first_day, top_critic_frac, pub_diversity, pub_entropy, low_activity_frac). LOO fit per-snap on noon-shifted reviews.

**Convention notes:**
- Snap is G2 (midnight UTC on close−N) for all three, so `snap_dbc_effective = N + midnight_utc_dbc`.
- Actuals are the same review count regardless of noon-shift (verified: shift only moves day-level reviews within-day, which doesn't change which reviews fall inside the phase-1 window).
- Skip rule (`first_review_dbc ≥ snap_dbc+1` AND `≥3 observed critics`) is applied uniformly.
- For the library variant, KDE training uses **raw** (non-shifted) reviews — matches library defaults. Observed-critic set / first_review_dbc come from the shifted global (invariant for phase-1).


In [ ]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
if NB_DIR.name != 'notebooks':
    NB_DIR = NB_DIR / 'notebooks'
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import pickle
import time

import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

import _helpers as H

print(f'cohort: {len(H.close_date_map)} resolved movies  ·  reviews: {len(H.reviews)}')


## Dual reviews setup (raw for library, shifted for ship + Ridge)

Keep a copy of the raw reviews for the library baseline. Apply noon-shift to `H.reviews` globally (so snapshot_state / actual_in_window / combined_score_with_scores all see the shifted version). Recompute `H.first_review_ts`, `H.gaps`, `H.gap_lookup` after the shift.


In [ ]:
reviews_raw = H.reviews.copy()

_day_mask = H.reviews['timestamp_confidence'] == 'd'
_n_shifted = int(_day_mask.sum())
H.reviews.loc[_day_mask, 'estimated_timestamp'] = (
    H.reviews.loc[_day_mask, 'estimated_timestamp'] + pd.Timedelta(hours=12)
)

H.first_review_ts = (
    H.reviews[H.reviews['movie_slug'].isin(H.close_date_map)]
    .groupby('movie_slug')['estimated_timestamp'].min()
)
_new_first = H.first_review_ts.to_dict()
H.gaps['first_review_ts'] = H.gaps['slug'].map(_new_first)
H.gaps['gap_days'] = (
    H.gaps['close_ts'] - H.gaps['first_review_ts']
).dt.total_seconds() / 86400
H.gap_lookup = dict(zip(H.gaps['slug'], H.gaps['gap_days']))

_activity = H.critic_activity_counts()
print(f'noon-shift applied: {_n_shifted} day-level reviews shifted')
print(f'reviews_raw kept separately for library baseline')
print(f'activity lookup built for Ridge features: {len(_activity)} critics')


## Configuration


In [ ]:
SNAP_DAYS_LIST = [5, 4, 3, 2, 1]
SHIP_ALPHA = 0.5
SIGMA_GAP = 8.0
N_TRAINING = 20
BANDWIDTH_FLOOR = 0.5
BANDWIDTH_CEILING_SHIP = 0.7
BANDWIDTH_CEILING_LIBRARY = 1e6  # effectively no cap (library default)
SHRINKAGE_K = 3.0
SCALING_THRESHOLD = 40.0
SCALING_CLAMP = (0.5, 2.0)
MIN_OBS_CRITICS = 3
MIN_TRAINING_SCORES = 5
RIDGE_ALPHA = 10.0
CHECKPOINT_EVERY = 300

FEATURES = [
    'observed_count', 'first_review_dbc', 'target_gap', 'observed_rate',
    'rate_last_day', 'rate_first_day', 'top_critic_frac',
    'pub_diversity', 'pub_entropy', 'low_activity_frac',
]

CACHE_PATH = H.CACHE_DIR / 'phase1_three_way.pkl'
print(f'cache: {CACHE_PATH}')


## Per-target-snap context

Compute shared state (snap_time, snap_dbc_effective, midnight_utc_dbc, observed state, actual phase-1) for a target-snap. All three variants use the same context.


In [ ]:
def build_context(target_slug, snap_days):
    close_ts = H.close_date_map[target_slug]
    midnight_utc = close_ts.floor('D')
    midnight_utc_dbc = (close_ts - midnight_utc).total_seconds() / 86400
    snap_time = midnight_utc - pd.Timedelta(days=snap_days)
    snap_dbc_effective = (close_ts - snap_time).total_seconds() / 86400

    state = H.snapshot_state(target_slug, snap_time)
    if state is None:
        return None
    if state['first_review_dbc'] < snap_dbc_effective + 1.0:
        return None
    if len(state['observed_critics']) < MIN_OBS_CRITICS:
        return None

    target_gap = H.gap_lookup.get(target_slug)
    if target_gap is None:
        return None

    target_window_days = state['first_review_dbc'] - snap_dbc_effective
    if target_window_days <= 0:
        return None

    actual = H.actual_in_window(target_slug, snap_dbc_effective, midnight_utc_dbc)

    return {
        'close_ts': close_ts, 'midnight_utc': midnight_utc,
        'midnight_utc_dbc': midnight_utc_dbc,
        'snap_time': snap_time, 'snap_dbc_effective': snap_dbc_effective,
        'state': state, 'target_gap': target_gap,
        'target_window_days': target_window_days, 'actual': actual,
    }


## Library-baseline prediction (v0.1 API)


In [ ]:
def predict_library(target_slug, ctx):
    """A1 + C1 + E1 + D2-floor-no-ceiling + F1. Uses raw (non-shifted) reviews for KDE training."""
    try:
        training_slugs = H.default_training_slugs(
            H.movies, exclude_slug=target_slug, n=N_TRAINING,
            before_date=ctx['close_ts'],
        )
        if len(training_slugs) < MIN_TRAINING_SCORES:
            return None
        profiles = H.build_critic_profiles(
            reviews_raw, H.close_date_map, training_slugs, verbose=False,
        )
        # Use the _capped helper with a huge ceiling = effectively no ceiling (library default).
        model = H.build_kde_lambda_model_capped(
            profiles, shrinkage_k=SHRINKAGE_K,
            bandwidth_floor=BANDWIDTH_FLOOR,
            bandwidth_ceiling=BANDWIDTH_CEILING_LIBRARY,
        )
        pred = H.predict_window_custom(
            model,
            dbc_from=ctx['snap_dbc_effective'],
            dbc_to=ctx['midnight_utc_dbc'],
            observed_critics=ctx['state']['observed_critics'],
            observed_count=ctx['state']['observed_count'],
            first_review_dbc=ctx['state']['first_review_dbc'],
            scaling_threshold=SCALING_THRESHOLD,
            scaling_clamp=SCALING_CLAMP,
        )
        if not np.isfinite(pred):
            return None
        return float(pred)
    except Exception:
        return None


## Ship-stack prediction (A3 + C2 + D2+D3 + E2 + F1 + G2 + H2 + I2 + K1)


In [ ]:
def predict_ship(target_slug, ctx):
    """Ship stack with noon-shifted reviews."""
    try:
        scores = H.combined_score_with_scores(
            target=target_slug, target_gap=ctx['target_gap'],
            target_critics=ctx['state']['observed_critics'],
            target_window_days=ctx['target_window_days'],
            k=N_TRAINING, alpha=SHIP_ALPHA, sigma_gap=SIGMA_GAP,
        )
        if len(scores) < MIN_TRAINING_SCORES:
            return None
        profiles = H.build_weighted_critic_profiles(
            H.reviews, H.close_date_map, scores,
        )
        model = H.build_weighted_kde_lambda_model(
            profiles, shrinkage_k=SHRINKAGE_K,
            bandwidth_floor=BANDWIDTH_FLOOR,
            bandwidth_ceiling=BANDWIDTH_CEILING_SHIP,
        )
        pred = H.predict_window_custom(
            model,
            dbc_from=ctx['snap_dbc_effective'],
            dbc_to=ctx['midnight_utc_dbc'],
            observed_critics=ctx['state']['observed_critics'],
            observed_count=ctx['state']['observed_count'],
            first_review_dbc=ctx['state']['first_review_dbc'],
            scaling_threshold=SCALING_THRESHOLD,
            scaling_clamp=SCALING_CLAMP,
        )
        if not np.isfinite(pred):
            return None
        return float(pred)
    except Exception:
        return None


## Ridge feature extraction

Features from the observation window `[first_review_ts, snap_time]`. Uses noon-shifted reviews (consistent with the ship stack). `rate_last_day` counts reviews in the final 24h before snap; `rate_first_day` counts in the first 24h post-first-review.


In [ ]:
def extract_ridge_features(target_slug, ctx):
    try:
        state = ctx['state']
        obs_window_days = ctx['target_window_days']

        mr = H.reviews[H.reviews['movie_slug'] == target_slug]
        obs_reviews = mr[
            (mr['estimated_timestamp'] < ctx['snap_time'])
            & (mr['estimated_timestamp'] < ctx['close_ts'])
        ]
        if len(obs_reviews) < MIN_OBS_CRITICS:
            return None

        first_review_ts_target = obs_reviews['estimated_timestamp'].min()

        stats = H.observed_review_stats(
            target_slug, first_review_ts_target, obs_window_days, _activity,
        )

        last_day_start = ctx['snap_time'] - pd.Timedelta(days=1)
        rate_last_day = int(
            ((obs_reviews['estimated_timestamp'] >= last_day_start)
             & (obs_reviews['estimated_timestamp'] < ctx['snap_time'])).sum()
        )

        first_day_end = first_review_ts_target + pd.Timedelta(days=1)
        rate_first_day = int(
            ((obs_reviews['estimated_timestamp'] >= first_review_ts_target)
             & (obs_reviews['estimated_timestamp'] < first_day_end)).sum()
        )

        return {
            'observed_count': state['observed_count'],
            'first_review_dbc': state['first_review_dbc'],
            'target_gap': ctx['target_gap'],
            'observed_rate': state['observed_count'] / obs_window_days,
            'rate_last_day': rate_last_day,
            'rate_first_day': rate_first_day,
            'top_critic_frac': stats['top_critic_frac'],
            'pub_diversity': stats['pub_diversity'],
            'pub_entropy': stats['pub_entropy'],
            'low_activity_frac': stats['low_activity_frac'],
        }
    except Exception:
        return None


## Smoke test on one target


In [ ]:
_smoke_slugs = [s for s in H.close_date_map
                if s not in {'the_drama', 'the_super_mario_galaxy_movie'}]
_smoke_target = sorted(_smoke_slugs, key=lambda s: H.close_date_map[s], reverse=True)[0]
print(f'smoke: {_smoke_target}   close={H.close_date_map[_smoke_target]}')
for snap_days in [5, 3, 1]:
    ctx = build_context(_smoke_target, snap_days)
    if ctx is None:
        print(f'  T-{snap_days}d: skipped')
        continue
    lib_pred = predict_library(_smoke_target, ctx)
    ship_pred = predict_ship(_smoke_target, ctx)
    feats = extract_ridge_features(_smoke_target, ctx)
    print(f'  T-{snap_days}d  actual={ctx["actual"]:3d}  '
          f'lib={lib_pred:6.2f}  ship={ship_pred:6.2f}  '
          f'ridge_features_ok={feats is not None}')


## Main sweep

Pass 1: for each (target × snap), build context, compute library_pred, ship_pred, actual, and Ridge feature vector. Pass 2: for each snap, fit LOO Ridge on the valid target set.


In [ ]:
def run_sweep(force=False):
    if CACHE_PATH.exists() and not force:
        with open(CACHE_PATH, 'rb') as f:
            cached = pickle.load(f)
        print(f'loaded cached {len(cached)} rows')
        return cached

    all_targets = sorted(H.close_date_map.keys())
    rows = []
    start = time.time()
    total = len(all_targets) * len(SNAP_DAYS_LIST)
    done = 0

    for target_slug in all_targets:
        for snap_days in SNAP_DAYS_LIST:
            ctx = build_context(target_slug, snap_days)
            done += 1
            if ctx is None:
                continue

            lib_pred = predict_library(target_slug, ctx)
            ship_pred = predict_ship(target_slug, ctx)
            feats = extract_ridge_features(target_slug, ctx)

            row = {
                'target_slug': target_slug,
                'snap_days': snap_days,
                'actual': int(ctx['actual']),
                'target_gap': float(ctx['target_gap']),
                'observed_count': ctx['state']['observed_count'],
                'first_review_dbc': ctx['state']['first_review_dbc'],
                'lib_pred': float(lib_pred) if lib_pred is not None else np.nan,
                'ship_pred': float(ship_pred) if ship_pred is not None else np.nan,
            }
            if feats is not None:
                row.update(feats)
            rows.append(row)

            if done % CHECKPOINT_EVERY == 0:
                elapsed = time.time() - start
                eta = elapsed / done * (total - done)
                print(f'  {done}/{total}  kept {len(rows)}  '
                      f'elapsed {elapsed/60:.1f}m  eta {eta/60:.1f}m')
                with open(CACHE_PATH, 'wb') as f:
                    pickle.dump(pd.DataFrame(rows), f)

    df = pd.DataFrame(rows)

    # Pass 2: per-snap LOO Ridge
    df['ridge_pred'] = np.nan
    for snap_days in SNAP_DAYS_LIST:
        snap_rows = df[df['snap_days'] == snap_days].copy()
        usable = snap_rows.dropna(subset=FEATURES + ['actual'])
        if len(usable) < 10:
            continue
        X = usable[FEATURES].values
        y = usable['actual'].values.astype(float)
        idx = usable.index.values
        preds = np.zeros(len(usable))
        for i in range(len(usable)):
            mask = np.ones(len(usable), dtype=bool)
            mask[i] = False
            m = Ridge(alpha=RIDGE_ALPHA)
            m.fit(X[mask], y[mask])
            preds[i] = m.predict(X[i:i+1])[0]
        df.loc[idx, 'ridge_pred'] = preds
        print(f'  ridge LOO T-{snap_days}d: fit {len(usable)} models')

    with open(CACHE_PATH, 'wb') as f:
        pickle.dump(df, f)
    print(f'saved {len(df)} rows to {CACHE_PATH}')
    return df

df = run_sweep()
print(f'total rows: {len(df)}')
df.head()


## Per-variant, per-snap summary


In [ ]:
def variant_metrics(sub, pred_col):
    s = sub.dropna(subset=[pred_col]).copy()
    if len(s) == 0:
        return None
    err = s[pred_col].values - s['actual'].values
    abs_err = np.abs(err)
    pred = s[pred_col].values
    actual = s['actual'].values.astype(float)
    safe_actual = np.where(actual > 0, actual, np.nan)
    return {
        'n': len(s), 'MAE': float(abs_err.mean()),
        'me': float(err.mean()),
        'med_err': float(np.median(err)),
        'med_abs_err': float(np.median(abs_err)),
        'p90_abs_err': float(np.quantile(abs_err, 0.9)),
        'med_ratio': float(np.nanmedian(pred / safe_actual)),
    }


PRED_COLS = [('library', 'lib_pred'), ('ship', 'ship_pred'), ('ridge', 'ridge_pred')]

print('=== per-variant summary ===\n')
summary_rows = []
for snap_days in SNAP_DAYS_LIST:
    sub = df[df['snap_days'] == snap_days]
    print(f'T-{snap_days}d')
    for name, col in PRED_COLS:
        m = variant_metrics(sub, col)
        if m is None:
            print(f'  {name:10s}  (no data)')
            continue
        summary_rows.append({'snap_days': snap_days, 'variant': name, **m})
        print(f'  {name:10s}  n={m["n"]:3d}  MAE={m["MAE"]:6.2f}  '
              f'me={m["me"]:+6.2f}  med_err={m["med_err"]:+6.2f}  '
              f'med|e|={m["med_abs_err"]:6.2f}  p90|e|={m["p90_abs_err"]:6.2f}  '
              f'med_ratio={m["med_ratio"]:5.2f}')
    print()
summary = pd.DataFrame(summary_rows)


## Side-by-side Δ vs library baseline


In [ ]:
print(f'{"snap":<6}{"variant":<10}{"MAE":>8}{"Δ vs lib":>12}{"% vs lib":>12}')
for snap_days in SNAP_DAYS_LIST:
    lib_row = summary[(summary['snap_days'] == snap_days) & (summary['variant'] == 'library')]
    if lib_row.empty:
        continue
    lib_mae = lib_row.iloc[0]['MAE']
    for name, col in PRED_COLS:
        m = summary[(summary['snap_days'] == snap_days) & (summary['variant'] == name)]
        if m.empty:
            continue
        mae = m.iloc[0]['MAE']
        if name == 'library':
            print(f'T-{snap_days}d  {name:<10}{mae:>8.2f}{"—":>12}{"—":>12}')
        else:
            delta = lib_mae - mae
            pct = 100 * delta / lib_mae if lib_mae else float('nan')
            print(f'T-{snap_days}d  {name:<10}{mae:>8.2f}{delta:>+12.2f}{pct:>+11.2f}%')
    print()


## Paired bootstrap CIs (all pairs)


In [ ]:
from itertools import combinations

PAIR_ORDER = [('library', 'ship'), ('library', 'ridge'), ('ship', 'ridge')]

print('=== paired bootstrap ΔMAE (A minus B, positive = A has higher MAE / B wins) ===\n')
print(f'{"snap":<6}{"A":<10}{"B":<10}{"Δ units":>10}{"CI95_lo":>10}{"CI95_hi":>10}'
      f'{"Δ %":>10}{"n":>6}  result')

ci_rows = []
for snap_days in SNAP_DAYS_LIST:
    snap_df = df[df['snap_days'] == snap_days]
    for a_name, b_name in PAIR_ORDER:
        a_col = {'library': 'lib_pred', 'ship': 'ship_pred', 'ridge': 'ridge_pred'}[a_name]
        b_col = {'library': 'lib_pred', 'ship': 'ship_pred', 'ridge': 'ridge_pred'}[b_name]
        paired = snap_df.dropna(subset=[a_col, b_col, 'actual']).copy()
        if len(paired) < 5:
            continue
        a_abs = np.abs(paired[a_col].values - paired['actual'].values)
        b_abs = np.abs(paired[b_col].values - paired['actual'].values)
        # delta = |A_err| − |B_err|. positive → B wins.
        deltas = a_abs - b_abs
        point, lo, hi = H.bootstrap_mae_delta(deltas, n_boot=1000)
        a_mae = a_abs.mean()
        pct = 100 * point / a_mae if a_mae else float('nan')
        if lo > 0:
            result = f'{b_name:>6} wins'
        elif hi < 0:
            result = f'{a_name:>6} wins'
        else:
            result = '    ns'
        ci_rows.append({
            'snap_days': snap_days, 'A': a_name, 'B': b_name,
            'delta': point, 'ci_lo': lo, 'ci_hi': hi,
            'delta_pct': pct, 'n': len(paired), 'result': result,
        })
        print(f'T-{snap_days}d  {a_name:<10}{b_name:<10}{point:>+10.3f}{lo:>+10.3f}{hi:>+10.3f}'
              f'{pct:>+10.2f}{len(paired):>6}  {result}')
    print()
ci_df = pd.DataFrame(ci_rows)


## Plot: MAE by snap by variant


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(10, 4))
colors = {'library': 'tab:gray', 'ship': 'tab:red', 'ridge': 'tab:blue'}
markers = {'library': 's', 'ship': 'o', 'ridge': '^'}
for name, _ in PRED_COLS:
    sub = summary[summary['variant'] == name].sort_values('snap_days', ascending=False)
    ax.plot(sub['snap_days'], sub['MAE'], '-',
            marker=markers[name], color=colors[name], label=name, markersize=8)
ax.invert_xaxis()
ax.set_xlabel('snap days before close')
ax.set_ylabel('MAE (reviews)')
ax.set_title('Phase-1 MAE by snap — three-way comparison')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Plot: bias (mean error) by snap by variant


In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
for name, _ in PRED_COLS:
    sub = summary[summary['variant'] == name].sort_values('snap_days', ascending=False)
    ax.plot(sub['snap_days'], sub['me'], '-',
            marker=markers[name], color=colors[name], label=name, markersize=8)
ax.axhline(0, color='black', linewidth=0.8, alpha=0.5)
ax.invert_xaxis()
ax.set_xlabel('snap days before close')
ax.set_ylabel('mean error (pred − actual)')
ax.set_title('Phase-1 bias by snap — three-way comparison')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## Observations

*(fill in after run)*
